In [1]:
!pip install kaggle

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.
Path to dataset files: /kaggle/input/chest-xray-pneumonia


In [3]:
!pip install tqdm

In [4]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tqdm import tqdm

# 1. PATH VA PARAMETRLARNI SOZLASH
# Kagglehub taqdim etgan pathni mana shu yerga nusxalang:
DATASET_PATH = "/kaggle/input/chest-xray-pneumonia/chest_xray/"

TRAIN_DIR = os.path.join(DATASET_PATH, 'train')
TEST_DIR = os.path.join(DATASET_PATH, 'test')

IMG_HEIGHT = 150
IMG_WIDTH = 150
BATCH_SIZE = 32
EPOCHS = 5

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE, class_mode='binary'
)
test_generator = test_datagen.flow_from_directory(
    TEST_DIR, target_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE, class_mode='binary', shuffle=False
)

model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.4),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("\n--- O'qitish jarayoni boshlanmoqda ---")

num_train_steps = len(train_generator)
num_test_steps = len(test_generator)

for epoch in range(EPOCHS):
    print(f"\nEpoxa {epoch+1}/{EPOCHS}")

    train_bar = tqdm(total=num_train_steps, desc="Training", unit="batch", bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]")

    epoch_loss = 0
    epoch_acc = 0

    for step in range(num_train_steps):
        x_batch, y_batch = next(train_generator)
        metrics = model.train_on_batch(x_batch, y_batch)

        epoch_loss += metrics[0]
        epoch_acc += metrics[1]

        train_bar.set_postfix({"Loss": f"{metrics[0]:.4f}", "Acc": f"{metrics[1]*100:.1f}%"})
        train_bar.update(1)

    train_bar.close()

    test_loss = 0
    test_acc = 0
    test_bar = tqdm(total=num_test_steps, desc="Testing ", unit="batch", leave=False)

    for step in range(num_test_steps):
        x_test, y_test = next(test_generator)
        t_metrics = model.test_on_batch(x_test, y_test)
        test_loss += t_metrics[0]
        test_acc += t_metrics[1]
        test_bar.update(1)

    test_bar.close()

    print(f"-> Yakuniy Train Aniqligi: {(epoch_acc/num_train_steps)*100:.2f}% | Train Yo'qotish: {epoch_loss/num_train_steps:.4f}")
    print(f"-> Yakuniy Test Aniqligi:  {(test_acc/num_test_steps)*100:.2f}% | Test Yo'qotish:  {test_loss/num_test_steps:.4f}")

model.save('tibbiy_model_pnevmoniya.h5')
print("\n[MUVAFFAQIYATLI]: Model 'tibbiy_model_pnevmoniya.h5' fayliga saqlandi!")

Found 5216 images belonging to 2 classes.
Found 624 images belonging to 2 classes.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



--- O'qitish jarayoni boshlanmoqda ---

Epoxa 1/5


Training: 100%|██████████| 163/163 [03:22<00:00,  1.25s/batch]


-> Yakuniy Train Aniqligi: 74.75% | Train Yo'qotish: 0.5597
-> Yakuniy Test Aniqligi:  79.97% | Test Yo'qotish:  0.4443

Epoxa 2/5


...

-> Yakuniy Train Aniqligi: 85.18% | Train Yo'qotish: 0.3540
-> Yakuniy Test Aniqligi:  85.53% | Test Yo'qotish:  0.3495

Epoxa 4/5


Training: 100%|██████████| 163/163 [02:57<00:00,  1.09s/batch]


-> Yakuniy Train Aniqligi: 86.51% | Train Yo'qotish: 0.3282
-> Yakuniy Test Aniqligi:  86.77% | Test Yo'qotish:  0.3242

Epoxa 5/5


Training: 100%|██████████| 163/163 [03:06<00:00,  1.14s/batch]


-> Yakuniy Train Aniqligi: 87.42% | Train Yo'qotish: 0.3092
-> Yakuniy Test Aniqligi:  87.61% | Test Yo'qotish:  0.3057

[MUVAFFAQIYATLI]: Model 'tibbiy_model_pnevmoniya.h5' fayliga saqlandi!
